# 00. Colab QAT Train Candidates v1

이 노트북은 15번 QAT 실험의 첫 단계다.

목표는 12번에서 만든 `best.pth`를 출발점으로 삼아, INT8 양자화 오차에 적응한 후보 `.pth` 4개를 만드는 것이다.

| 후보 | QAT fake quant 적용 범위 | 의도 |
|---|---|---|
| `qat_layer4_only` | ResNet18 `layer4` | 11b에서 가장 아까웠던 near-miss 범위부터 확인 |
| `qat_backbone_only` | ResNet18 backbone 전체 | feature extractor 전체를 INT8 친화적으로 적응 |
| `qat_backbone_neck_only` | backbone + Aggregator neck | head는 보호하고 feature/neck까지 적응 |
| `qat_full_model` | backbone + neck + head | 가장 공격적인 후보. 성공하면 속도 이득 가능성이 크지만 위험도 큼 |

여기서 만드는 `.pth`는 아직 INT8 배포 모델이 아니다. 정확히는 **ONNX Runtime INT8 양자화에 잘 견디도록 fine-tuning된 FP32 checkpoint**다.

## QAT-lite라고 부르는 이유

정석 PyTorch QAT는 보통 `prepare_qat -> train -> convert`로 PyTorch quantized model을 만든다. 하지만 우리의 최종 배포 대상은 PyTorch runtime이 아니라 **ONNX Runtime on Raspberry Pi**다.

그래서 이 노트북은 다음 방식을 쓴다.

```text
best.pth
→ fake quantization을 학습 중에 주입
→ QAT-adapted FP32 .pth
→ 다음 노트북에서 ONNX export
→ ORT static quantization
→ INT8 ONNX
```

즉 QAT의 핵심인 “학습 중 양자화 오차에 적응”은 수행하되, 최종 quantized graph 생성은 ONNX Runtime quantizer에게 맡긴다.

## 0. Colab 입력 파일

새 데이터셋은 만들지 않는다. 12번 full fine-tuning에 사용한 Field1+2 train/val dataset을 그대로 사용한다.

- `map_culane_localfit_train_field1_field2_v1.tar.gz`
- 12번 full fine-tuning 결과의 `best.pth`
- 공식 CLRKDNet repo

`field3`와 `holdout/background`는 여기서 학습에 쓰지 않는다. 그 둘은 02/03 검증 단계에 남겨둔다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, math, os, shutil, subprocess, sys, tarfile, textwrap, time

DRIVE_DIR = Path('/content/drive/MyDrive/Colab Notebooks/26-1학기_임베디드인공지능시스템최적화/07_CLRKDNet_QAT')
REPO_DIR = Path('/content/CLRKDNet')
DATA_ROOT = REPO_DIR / 'data'

DATASET_NAME = 'map_culane_localfit_train_field1_field2_v1'
DATASET_TAR = DRIVE_DIR / f'{DATASET_NAME}.tar.gz'
DATASET_DIR = DATA_ROOT / DATASET_NAME

OUT_DRIVE = DRIVE_DIR / 'outputs_qat_v1'
OUT_DRIVE.mkdir(parents=True, exist_ok=True)

QAT_EPOCHS = 2
QAT_LR = 2e-5
BATCH_SIZE = 8
TRAIN_ROWS = 6894
RUN_PREFIX = 'MapLane_QAT_Field12_v1'

QAT_CANDIDATES = [
    dict(name='qat_layer4_only', scope='layer4_only', enabled=True),
    dict(name='qat_backbone_only', scope='backbone_only', enabled=True),
    dict(name='qat_backbone_neck_only', scope='backbone_neck_only', enabled=True),
    dict(name='qat_full_model', scope='full_model', enabled=True),
]

print('DRIVE_DIR:', DRIVE_DIR)
print('DATASET_TAR:', DATASET_TAR, DATASET_TAR.exists())
print('OUT_DRIVE:', OUT_DRIVE)
assert DATASET_TAR.exists(), f'missing dataset tar: {DATASET_TAR}'

In [ ]:
def find_best_pth():
    explicit_candidates = [
        DRIVE_DIR / 'MapLane_LocalFit_Field12_v1_best.pth',
        DRIVE_DIR / 'best.pth',
    ]
    for p in explicit_candidates:
        if p.exists():
            return p

    search_roots = [DRIVE_DIR / 'outputs_localfit_v1', DRIVE_DIR]
    found = []
    for root in search_roots:
        if root.exists():
            found.extend(root.rglob('best.pth'))
    found = sorted(set(found), key=lambda p: p.stat().st_mtime, reverse=True)
    if not found:
        raise FileNotFoundError('12번 best.pth를 찾지 못했다. DRIVE_DIR 아래에 best.pth 또는 outputs_localfit_v1/checkpoint_mirror/.../best.pth가 필요하다.')
    print('best.pth candidates:')
    for p in found[:8]:
        print(' ', p, 'mtime=', time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(p.stat().st_mtime)))
    return found[0]

BEST_PTH_SRC = find_best_pth()
BEST_PTH_DST = REPO_DIR / 'MapLane_LocalFit_Field12_v1_best.pth'
print('selected BEST_PTH_SRC:', BEST_PTH_SRC)

## 1. 공식 Repo 설치와 12번 호환성 패치

05번 학습 노트북과 같은 원칙을 따른다.

- 공식 CLRKDNet 학습/손실 구조는 보존한다.
- Colab에서 깨지는 `mmcv`, `torchvision`, `np.sctypes` 호환성만 패치한다.
- validation NMS CUDA op가 없어도 학습이 중단되지 않도록 fallback NMS를 둔다.
- CULane metric의 hard-coded 1640x590/270~590 가정을 프로젝트 map geometry에 맞게 패치한다.

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/weiqingq/CLRKDNet.git', str(REPO_DIR)], check=True)
else:
    print('repo already exists:', REPO_DIR)

import importlib
import numpy as np
if not hasattr(np, 'sctypes'):
    np.sctypes = {
        'float': [np.float16, np.float32, np.float64],
        'int': [np.int8, np.int16, np.int32, np.int64],
        'uint': [np.uint8, np.uint16, np.uint32, np.uint64],
        'complex': [np.complex64, np.complex128],
        'others': [np.bool_, np.object_, np.bytes_, np.str_],
    }

checks = [
    ('addict', 'addict'),
    ('yapf', 'yapf==0.40.1'),
    ('pathspec', 'pathspec'),
    ('timm', 'timm'),
    ('pytorch_warmup', 'pytorch_warmup'),
    ('ptflops', 'ptflops'),
    ('imgaug', 'imgaug'),
    ('shapely', 'shapely'),
    ('p_tqdm', 'p_tqdm'),
]
missing = []
for module_name, pip_name in checks:
    try:
        importlib.import_module(module_name)
    except Exception:
        missing.append(pip_name)
if missing:
    print('installing:', missing)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
else:
    print('all optional packages already available')
print('numpy:', np.__version__)

In [ ]:
def write_text(path: Path, text: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(text).lstrip(), encoding='utf-8')

write_text(REPO_DIR / 'clrkd' / 'datasets' / '__init__.py', '''
from .registry import build_dataset, build_dataloader
from .culane import CULane
from .process import *
''')

base_dataset_py = REPO_DIR / 'clrkd' / 'datasets' / 'base_dataset.py'
base_text = base_dataset_py.read_text(encoding='utf-8')
base_text = base_text.replace('import torchvision\n', '')
base_dataset_py.write_text(base_text, encoding='utf-8')

write_text(REPO_DIR / 'mmcv' / '__init__.py', '''
__version__ = 'local-shim'

def jit(*jit_args, **jit_kwargs):
    def decorator(func):
        return func
    if len(jit_args) == 1 and callable(jit_args[0]) and not jit_kwargs:
        return jit_args[0]
    return decorator

def load(filename, *args, **kwargs):
    import json
    with open(filename, 'r') as f:
        return json.load(f)

def dump(obj, file=None, file_format=None, *args, **kwargs):
    import json
    text = json.dumps(obj, indent=2)
    if file is None:
        return text
    with open(file, 'w') as f:
        f.write(text)
''')

write_text(REPO_DIR / 'mmcv' / 'cnn' / '__init__.py', '''
import torch.nn as nn

class ConvModule(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0,
                 dilation=1, groups=1, bias='auto', conv_cfg=None, norm_cfg=None,
                 act_cfg=dict(type='ReLU'), inplace=True, **kwargs):
        layers = []
        use_bias = bias if isinstance(bias, bool) else (norm_cfg is None)
        layers.append(nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride,
                                padding=padding, dilation=dilation, groups=groups, bias=use_bias))
        if norm_cfg is not None:
            layers.append(nn.BatchNorm2d(out_channels))
        if act_cfg is not None:
            layers.append(nn.ReLU(inplace=inplace))
        super().__init__(*layers)
''')

write_text(REPO_DIR / 'mmcv' / 'parallel' / '__init__.py', '''
import torch
from torch.utils.data._utils.collate import default_collate

class DataContainer:
    def __init__(self, data, stack=False, padding_value=0, cpu_only=False, pad_dims=2):
        self.data = data
        self.stack = stack
        self.padding_value = padding_value
        self.cpu_only = cpu_only
        self.pad_dims = pad_dims

def collate(batch, samples_per_gpu=1):
    if not batch:
        return batch
    elem = batch[0]
    if isinstance(elem, DataContainer):
        return [b.data for b in batch] if elem.cpu_only else default_collate([b.data for b in batch])
    if isinstance(elem, dict):
        return {key: collate([d[key] for d in batch], samples_per_gpu) for key in elem}
    if isinstance(elem, (list, tuple)):
        transposed = list(zip(*batch))
        return [collate(samples, samples_per_gpu) for samples in transposed]
    try:
        return default_collate(batch)
    except Exception:
        return batch

class MMDataParallel(torch.nn.DataParallel):
    def __init__(self, module, device_ids=None, dim=0):
        if device_ids is None:
            device_ids = [0] if torch.cuda.is_available() else []
        super().__init__(module, device_ids=device_ids, dim=dim)
''')

for path in REPO_DIR.rglob('*.py'):
    text = path.read_text(encoding='utf-8')
    new = text.replace('collections.Iterable', 'collections.abc.Iterable')
    if new != text:
        if 'import collections.abc' not in new:
            new = new.replace('import collections\n', 'import collections\nimport collections.abc\n')
        path.write_text(new, encoding='utf-8')

write_text(REPO_DIR / 'sitecustomize.py', '''
import numpy as np
if not hasattr(np, 'sctypes'):
    np.sctypes = {
        'float': [np.float16, np.float32, np.float64],
        'int': [np.int8, np.int16, np.int32, np.int64],
        'uint': [np.uint8, np.uint16, np.uint32, np.uint64],
        'complex': [np.complex64, np.complex128],
        'others': [np.bool_, np.object_, np.bytes_, np.str_],
    }
''')

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
import importlib, mmcv
mmcv = importlib.reload(mmcv)
for name in ['jit', 'load', 'dump']:
    assert hasattr(mmcv, name), f'mmcv shim missing {name}'
print('compatibility patches written and verified')

In [ ]:
write_text(REPO_DIR / 'clrkd' / 'ops' / 'nms.py', '''
import torch

def _mean_abs_lane_distance(a, b):
    ax = a[5:]
    bx = b[5:]
    valid = torch.isfinite(ax) & torch.isfinite(bx) & (ax >= 0) & (bx >= 0)
    if int(valid.sum().item()) < 2:
        return torch.tensor(float('inf'), device=a.device)
    return torch.mean(torch.abs(ax[valid] - bx[valid]))

def nms(boxes, scores, overlap=50, top_k=4):
    if boxes is None or scores is None or scores.numel() == 0:
        keep = torch.empty((0,), dtype=torch.long, device=scores.device if scores is not None else 'cpu')
        return keep, 0, None
    order = torch.argsort(scores, descending=True)
    kept = []
    for idx in order:
        if len(kept) >= int(top_k):
            break
        duplicate = False
        for kept_idx in kept:
            if _mean_abs_lane_distance(boxes[idx], boxes[kept_idx]) <= overlap:
                duplicate = True
                break
        if not duplicate:
            kept.append(idx)
    keep = torch.stack(kept).long() if kept else torch.empty((0,), dtype=torch.long, device=scores.device)
    return keep, int(keep.numel()), None
''')

culane_py = REPO_DIR / 'clrkd' / 'datasets' / 'culane.py'
text = culane_py.read_text(encoding='utf-8')
ys_target = 'ys = np.arange(self.cfg.cut_height, self.cfg.ori_img_h, 8) / self.cfg.ori_img_h'
if ys_target not in text:
    applied = False
    for old in ['ys = np.arange(270, 590, 8) / self.cfg.ori_img_h', 'ys = np.arange(270, 590, 8) / 590']:
        if old in text:
            text = text.replace(old, ys_target)
            applied = True
    assert applied, 'culane.py sample-y source pattern not found'
assert ys_target in text
assert 'np.arange(270, 590, 8)' not in text

call_target = 'official=True, img_shape=(self.cfg.ori_img_h, self.cfg.ori_img_w, 3))'
if call_target not in text:
    assert text.count('official=True)') >= 2, 'expected official=True calls not found'
    text = text.replace('official=True)', call_target)
assert text.count(call_target) >= 2
culane_py.write_text(text, encoding='utf-8')

metric_py = REPO_DIR / 'clrkd' / 'utils' / 'culane_metric.py'
text = metric_py.read_text(encoding='utf-8')
sig_target = 'sequential=False,\n                     img_shape=(590, 1640, 3)):'
if sig_target not in text:
    assert 'sequential=False):' in text, 'culane_metric.py signature source pattern not found'
    text = text.replace('sequential=False):', sig_target)
if 'img_shape = (590, 1640, 3)' in text:
    text = text.replace('img_shape = (590, 1640, 3)', '# img_shape is provided by dataset config for project-map data')
metric_py.write_text(text, encoding='utf-8')
print('validation geometry patch applied')

## 2. 데이터셋과 best.pth 준비

QAT는 기존 학습 데이터셋을 그대로 사용한다. 여기서는 `field3`를 절대 학습에 섞지 않는다.

In [ ]:
DATA_ROOT.mkdir(parents=True, exist_ok=True)
if DATASET_DIR.exists():
    print('dataset already extracted:', DATASET_DIR)
else:
    print('extracting:', DATASET_TAR)
    with tarfile.open(DATASET_TAR, 'r:gz') as tf:
        tf.extractall(DATA_ROOT)
    print('extracted:', DATASET_DIR)

shutil.copy2(BEST_PTH_SRC, BEST_PTH_DST)
print('best checkpoint copied:', BEST_PTH_DST)

cache_dir = REPO_DIR / 'cache'
if cache_dir.exists():
    shutil.rmtree(cache_dir)
    print('removed stale cache:', cache_dir)

assert DATASET_DIR.exists()
assert BEST_PTH_DST.exists()

## 3. QAT config 생성

12번 full 학습 config를 그대로 두되, QAT fine-tuning에 맞게 아래만 바꾼다.

- `epochs`: 기본 2
- `lr`: 기본 `2e-5`
- `finetune_from`: 12번 `best.pth`
- `qat_scope`: 후보별 fake quant 적용 범위

validation metric은 학습 중 sanity check용이다. 최종 판단은 02 노트북의 official decoder + steering parity로 한다.

In [ ]:
def write_qat_config(path: Path, candidate: dict):
    batches_per_epoch = math.ceil(TRAIN_ROWS / BATCH_SIZE)
    total_iter = batches_per_epoch * QAT_EPOCHS
    work_name = f"{RUN_PREFIX}_{candidate['name']}"
    text = f'''
# Auto-generated by 15/00 QAT notebook.
# Candidate: {candidate['name']} / scope={candidate['scope']}

num_points = 72
max_lanes = 4
sample_y = range(971, 444, -20)
epochs = {QAT_EPOCHS}
batch_size = {BATCH_SIZE}
eval_ep = 1
save_ep = 1

iou_loss_weight = 2.
cls_loss_weight = 2.
xyt_loss_weight = 0.2
seg_loss_weight = 1.0
att_loss_weight = 1.0
log_loss_weight = 5.0
priors_loss_weight = 3.0

optimizer = dict(type='AdamW', lr={QAT_LR})
total_iter = {total_iter}
scheduler = dict(type='CosineAnnealingLR', T_max=total_iter)
test_parameters = dict(conf_threshold=0.35, nms_thres=50, nms_topk=max_lanes)
work_dirs = 'work_dirs/{work_name}'

net = dict(type='Detector')
backbone = dict(type='ResNetWrapper', resnet='resnet18', pretrained=False)
neck = dict(type='Aggregator', in_channels=[512], out_channels=64)
heads = dict(type='CLRHead', num_priors=192, refine_layers=1, fc_hidden_dim=64, sample_points=36)

img_norm = dict(mean=[103.939, 116.779, 123.68], std=[1., 1., 1.])
ori_img_w = 1296
ori_img_h = 972
img_w = 800
img_h = 320
cut_height = 445

train_process = [
    dict(type='GenerateLaneLine', transforms=[
        dict(name='Resize', parameters=dict(size=dict(height=img_h, width=img_w)), p=1.0),
        dict(name='HorizontalFlip', parameters=dict(p=1.0), p=0.5),
        dict(name='ChannelShuffle', parameters=dict(p=1.0), p=0.1),
        dict(name='MultiplyAndAddToBrightness', parameters=dict(mul=(0.85, 1.15), add=(-10, 10)), p=0.6),
        dict(name='AddToHueAndSaturation', parameters=dict(value=(-10, 10)), p=0.7),
        dict(name='OneOf', transforms=[
            dict(name='MotionBlur', parameters=dict(k=(3, 5))),
            dict(name='MedianBlur', parameters=dict(k=(3, 5)))], p=0.2),
        dict(name='Affine', parameters=dict(translate_percent=dict(x=(-0.1, 0.1), y=(-0.1, 0.1)), rotate=(-10, 10), scale=(0.8, 1.2)), p=0.7),
        dict(name='Resize', parameters=dict(size=dict(height=img_h, width=img_w)), p=1.0),
    ]),
    dict(type='ToTensor', keys=['img', 'lane_line', 'seg']),
]

val_process = [
    dict(type='GenerateLaneLine', transforms=[
        dict(name='Resize', parameters=dict(size=dict(height=img_h, width=img_w)), p=1.0),
    ], training=False),
    dict(type='ToTensor', keys=['img']),
]

dataset_path = './data/{DATASET_NAME}'
dataset_type = 'CULane'
diff_path = None
threshold = 15
dataset = dict(
    train=dict(type=dataset_type, data_root=dataset_path, split='train', processes=train_process),
    val=dict(type=dataset_type, data_root=dataset_path, split='val', processes=val_process),
    test=dict(type=dataset_type, data_root=dataset_path, split='test', processes=val_process),
)

workers = 2
log_interval = 20
seed = 0
num_classes = 4 + 1
ignore_label = 255
bg_weight = 0.4
lr_update_by_epoch = False

qat_scope = '{candidate['scope']}'
qat_candidate_name = '{candidate['name']}'
qat_options = dict(
    enabled=True,
    activation_quant='quint8_asymmetric',
    weight_quant='qint8_symmetric',
    activation_ema=0.95,
    quantize_eval=True,
)
'''
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(text).lstrip(), encoding='utf-8')
    return path

CONFIG_DIR = REPO_DIR / 'configs' / 'qat_field12_v1'
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
config_paths = {}
for cand in QAT_CANDIDATES:
    p = write_qat_config(CONFIG_DIR / f"{cand['name']}.py", cand)
    config_paths[cand['name']] = p
    print(cand['name'], '->', p)

## 4. Fake quantization patch

모델 구조를 wrapper module로 바꾸지 않고, 선택한 `Conv2d`/`Linear` module의 `forward`만 monkey-patch한다.

이렇게 하면 학습 중에는 fake quantization이 들어가지만, 저장되는 `.pth`의 weight key는 기존 CLRKDNet과 호환된다.

In [ ]:
write_text(REPO_DIR / 'clrkd' / 'utils' / 'qat_fake_quant.py', r'''
import types
import torch
import torch.nn as nn
import torch.nn.functional as F

def _to_float(v, default):
    try:
        return float(v)
    except Exception:
        return default

def _update_ema(module, x, ema=0.95):
    cur_min = float(x.detach().min().item())
    cur_max = float(x.detach().max().item())
    if not hasattr(module, '_qat_act_min') or module._qat_act_min is None:
        module._qat_act_min = cur_min
        module._qat_act_max = cur_max
    elif module.training:
        module._qat_act_min = ema * float(module._qat_act_min) + (1.0 - ema) * cur_min
        module._qat_act_max = ema * float(module._qat_act_max) + (1.0 - ema) * cur_max
    return float(module._qat_act_min), float(module._qat_act_max)

def fake_quant_activation(x, module, ema=0.95, enabled=True):
    if not enabled:
        return x
    x_min, x_max = _update_ema(module, x, ema)
    x_min = min(x_min, 0.0)
    x_max = max(x_max, 0.0)
    if x_max <= x_min + 1e-8:
        return x
    scale = max((x_max - x_min) / 255.0, 1e-8)
    zero_point = int(round(-x_min / scale))
    zero_point = max(0, min(255, zero_point))
    return torch.fake_quantize_per_tensor_affine(x, float(scale), int(zero_point), 0, 255)

def fake_quant_weight(w, enabled=True):
    if not enabled:
        return w
    max_abs = float(w.detach().abs().max().item())
    if max_abs <= 1e-12:
        return w
    scale = max(max_abs / 127.0, 1e-12)
    return torch.fake_quantize_per_tensor_affine(w, float(scale), 0, -128, 127)

def _clean_name(name):
    return name[7:] if name.startswith('module.') else name

def should_patch(name, scope):
    clean = _clean_name(name)
    if scope == 'layer4_only':
        return clean.startswith('backbone.model.layer4')
    if scope == 'backbone_only':
        return clean.startswith('backbone.')
    if scope == 'backbone_neck_only':
        return clean.startswith('backbone.') or clean.startswith('neck.')
    if scope == 'full_model':
        return clean.startswith('backbone.') or clean.startswith('neck.') or clean.startswith('heads.')
    raise ValueError('unknown qat scope: ' + str(scope))

def patch_conv2d(module, options):
    if getattr(module, '_qat_fake_quant_patched', False):
        return False
    ema = _to_float(options.get('activation_ema', 0.95), 0.95)
    quantize_eval = bool(options.get('quantize_eval', True))
    def forward(self, input):
        enabled = self.training or quantize_eval
        input_q = fake_quant_activation(input, self, ema=ema, enabled=enabled)
        weight_q = fake_quant_weight(self.weight, enabled=enabled)
        return F.conv2d(input_q, weight_q, self.bias, self.stride, self.padding, self.dilation, self.groups)
    module.forward = types.MethodType(forward, module)
    module._qat_fake_quant_patched = True
    module._qat_act_min = None
    module._qat_act_max = None
    return True

def patch_conv1d(module, options):
    if getattr(module, '_qat_fake_quant_patched', False):
        return False
    ema = _to_float(options.get('activation_ema', 0.95), 0.95)
    quantize_eval = bool(options.get('quantize_eval', True))
    def forward(self, input):
        enabled = self.training or quantize_eval
        input_q = fake_quant_activation(input, self, ema=ema, enabled=enabled)
        weight_q = fake_quant_weight(self.weight, enabled=enabled)
        return F.conv1d(input_q, weight_q, self.bias, self.stride, self.padding, self.dilation, self.groups)
    module.forward = types.MethodType(forward, module)
    module._qat_fake_quant_patched = True
    module._qat_act_min = None
    module._qat_act_max = None
    return True

def patch_linear(module, options):
    if getattr(module, '_qat_fake_quant_patched', False):
        return False
    ema = _to_float(options.get('activation_ema', 0.95), 0.95)
    quantize_eval = bool(options.get('quantize_eval', True))
    def forward(self, input):
        enabled = self.training or quantize_eval
        input_q = fake_quant_activation(input, self, ema=ema, enabled=enabled)
        weight_q = fake_quant_weight(self.weight, enabled=enabled)
        return F.linear(input_q, weight_q, self.bias)
    module.forward = types.MethodType(forward, module)
    module._qat_fake_quant_patched = True
    module._qat_act_min = None
    module._qat_act_max = None
    return True

def apply_qat_fake_quant(net, scope, options=None):
    options = options or {}
    patched = []
    for name, module in net.named_modules():
        if not should_patch(name, scope):
            continue
        did_patch = False
        if isinstance(module, nn.Conv2d):
            did_patch = patch_conv2d(module, options)
        elif isinstance(module, nn.Conv1d):
            did_patch = patch_conv1d(module, options)
        elif isinstance(module, nn.Linear):
            did_patch = patch_linear(module, options)
        if did_patch:
            patched.append(name)
    if not patched:
        raise RuntimeError('QAT fake quant patched zero modules for scope=' + str(scope))
    return patched
''')

runner_py = REPO_DIR / 'clrkd' / 'engine' / 'runner.py'
text = runner_py.read_text(encoding='utf-8')
needle = '        self.resume()\n        self.optimizer = build_optimizer(self.cfg, self.net)'
insert = '''        self.resume()\n        if hasattr(self.cfg, 'qat_scope') and self.cfg.qat_scope:\n            from clrkd.utils.qat_fake_quant import apply_qat_fake_quant\n            qat_options = self.cfg.qat_options if hasattr(self.cfg, 'qat_options') else {}\n            patched = apply_qat_fake_quant(self.net, self.cfg.qat_scope, qat_options)\n            self.recorder.logger.info('QAT fake quant scope: ' + str(self.cfg.qat_scope))\n            self.recorder.logger.info('QAT fake quant patched modules: ' + str(len(patched)))\n            self.recorder.logger.info('QAT fake quant first modules: ' + str(patched[:20]))\n        self.optimizer = build_optimizer(self.cfg, self.net)'''
if insert not in text:
    assert needle in text, 'runner.py patch point not found'
    text = text.replace(needle, insert)
runner_py.write_text(text, encoding='utf-8')
print('QAT fake quant module written and runner patched')

## 4-1. STOP CHECK: QAT scope가 의도한 module만 patch하는지 확인

여기서는 실제 학습을 시작하기 전에 후보별 patch 범위를 검사한다.

이 셀의 목적은 “layer4라고 생각했는데 다른 곳이 patch됨”, “full model인데 `Conv1d`가 빠짐” 같은 조용한 오류를 막는 것이다.

확인 기준:

- `qat_layer4_only`: `backbone.model.layer4.*`만 patch
- `qat_backbone_only`: `backbone.*`만 patch
- `qat_backbone_neck_only`: `backbone.*` 또는 `neck.*`만 patch, `heads.*` 금지
- `qat_full_model`: `backbone/neck/heads` 모두 patch 대상에 포함

In [ ]:
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import importlib
from clrkd.utils.config import Config
import clrkd.models
from clrkd.models.registry import build_net
from clrkd.utils.qat_fake_quant import apply_qat_fake_quant

def strip_module(name):
    return name[7:] if name.startswith('module.') else name

scope_rows = []
for cand in QAT_CANDIDATES:
    cfg_probe = Config.fromfile(str(config_paths[cand['name']]))
    model_probe = build_net(cfg_probe).eval()
    patched = [strip_module(x) for x in apply_qat_fake_quant(model_probe, cfg_probe.qat_scope, cfg_probe.qat_options)]
    row = dict(
        candidate=cand['name'],
        scope=cfg_probe.qat_scope,
        patched_count=len(patched),
        backbone=sum(x.startswith('backbone.') for x in patched),
        neck=sum(x.startswith('neck.') for x in patched),
        heads=sum(x.startswith('heads.') for x in patched),
        conv1d=sum('roi_gather.f_query' in x or 'roi_gather.W' in x for x in patched),
        first_modules=patched[:12],
    )
    scope_rows.append(row)

    if cand['scope'] == 'layer4_only':
        assert row['patched_count'] == 5, row
        assert all(x.startswith('backbone.model.layer4') for x in patched), patched
    elif cand['scope'] == 'backbone_only':
        assert row['patched_count'] == 20, row
        assert row['backbone'] == row['patched_count'] and row['neck'] == 0 and row['heads'] == 0, row
    elif cand['scope'] == 'backbone_neck_only':
        assert row['patched_count'] == 22, row
        assert row['backbone'] == 20 and row['neck'] == 2 and row['heads'] == 0, row
    elif cand['scope'] == 'full_model':
        assert row['backbone'] == 20 and row['neck'] == 2 and row['heads'] > 0, row
        assert row['conv1d'] >= 2, row
    else:
        raise AssertionError(cand)
    del model_probe

print(json.dumps(scope_rows, indent=2, ensure_ascii=False))
print('STOP CHECK passed: QAT scope patching matches intended candidates.')

## 5. STOP CHECK: 데이터와 모델 경로 확인

실제 train tensor가 05/09/10과 같은 `[0,1]` BGR인지 다시 확인한다. QAT가 잘 되어도 입력 contract가 흔들리면 전부 의미가 없다.

In [ ]:
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from clrkd.utils.config import Config
cfg = Config.fromfile(str(config_paths['qat_layer4_only']))

for module_name in list(sys.modules):
    if module_name == 'clrkd.datasets' or module_name.startswith('clrkd.datasets.'):
        del sys.modules[module_name]
import clrkd.datasets
from clrkd.datasets import build_dataset
from clrkd.datasets.registry import DATASETS, PROCESS

assert 'CULane' in DATASETS.module_dict
assert 'GenerateLaneLine' in PROCESS.module_dict
assert 'ToTensor' in PROCESS.module_dict
train_dataset = build_dataset(cfg.dataset.train, cfg)
sample = train_dataset[0]
img = sample['img'].data if hasattr(sample.get('img'), 'data') else sample['img']
import torch
img_tensor = img if isinstance(img, torch.Tensor) else torch.as_tensor(img)
print('img shape:', tuple(img_tensor.shape))
print('img min/max/mean:', float(img_tensor.min()), float(img_tensor.max()), float(img_tensor.float().mean()))
assert img_tensor.shape[-2:] == (320, 800)
assert float(img_tensor.max()) <= 1.01
assert float(img_tensor.min()) >= -0.01
print('STOP CHECK passed: training input remains BGR float [0,1].')

## 6. 후보별 QAT fine-tuning 실행

각 후보는 같은 `best.pth`에서 독립적으로 시작한다. 즉 후보 간에 이어서 학습하지 않는다.

출력은 Colab runtime의 `work_dirs/`와 Drive의 `outputs_qat_v1/` 양쪽에 남긴다.

In [ ]:
def latest_run_dir(work_name: str):
    root = REPO_DIR / 'work_dirs' / work_name
    if not root.exists():
        return None
    candidates = [p for p in root.iterdir() if p.is_dir()]
    return max(candidates, key=lambda p: p.stat().st_mtime) if candidates else None

def should_live_print(line: str) -> bool:
    if ' - INFO - epoch:' in line:
        import re
        m = re.search(r'step:\s*(\d+)', line)
        if not m:
            return True
        step = int(m.group(1))
        return step <= 5 or step % 100 == 1 or step % 862 in (0, 1)
    keep_tokens = [
        'QAT fake quant scope:',
        'QAT fake quant patched modules:',
        'Build train loader',
        'Start training',
        'Number of images loaded',
        'iou thr: 0.50',
        'metric:',
        'Best metric',
        'Traceback',
        'Error',
        'Exception',
    ]
    return any(token in line for token in keep_tokens)

def mirror_candidate_to_drive(cand_name: str, work_name: str):
    run_dir = latest_run_dir(work_name)
    if run_dir is None:
        return None
    mirror_dir = OUT_DRIVE / 'checkpoint_mirror' / work_name / run_dir.name
    mirror_dir.mkdir(parents=True, exist_ok=True)
    copied = []
    log_path = run_dir / 'log.txt'
    if log_path.exists():
        shutil.copy2(log_path, mirror_dir / 'log.txt')
        copied.append('log.txt')
    ckpt_dir = run_dir / 'ckpt'
    if ckpt_dir.exists():
        out_ckpt = mirror_dir / 'ckpt'
        out_ckpt.mkdir(parents=True, exist_ok=True)
        for src in sorted(ckpt_dir.glob('*.pth')):
            dst = out_ckpt / src.name
            if (not dst.exists()) or src.stat().st_size != dst.stat().st_size:
                shutil.copy2(src, dst)
            copied.append('ckpt/' + src.name)
    manifest = dict(
        candidate=cand_name,
        work_name=work_name,
        run_dir=str(run_dir),
        mirror_dir=str(mirror_dir),
        source_best_pth=str(BEST_PTH_SRC),
        qat_epochs=QAT_EPOCHS,
        qat_lr=QAT_LR,
        copied=copied,
        created_at=time.strftime('%Y-%m-%d %H:%M:%S'),
    )
    (mirror_dir / 'candidate_manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
    print(f'[mirror] {cand_name}: {copied[-6:]} -> {mirror_dir}', flush=True)
    return mirror_dir

def run_qat_candidate(candidate: dict):
    if not candidate.get('enabled', True):
        print('[skip]', candidate['name'])
        return None
    cand_name = candidate['name']
    config_path = config_paths[cand_name]
    work_name = f'{RUN_PREFIX}_{cand_name}'
    cmd = [
        'python', 'main.py', str(config_path),
        '--gpus', '0',
        '--finetune_from', str(BEST_PTH_DST),
        '--work_dirs', f'work_dirs/{work_name}',
    ]
    print('\n[run]', cand_name)
    print('command:', ' '.join(cmd), flush=True)
    env = os.environ.copy()
    env['PYTHONPATH'] = str(REPO_DIR) + (':' + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
    env['PYTHONUNBUFFERED'] = '1'
    local_log_dir = REPO_DIR / 'runtime_logs_qat'
    local_log_dir.mkdir(parents=True, exist_ok=True)
    local_debug_path = local_log_dir / f'{cand_name}_subprocess_output.txt'
    drive_debug_path = OUT_DRIVE / 'debug_logs' / f'{cand_name}_subprocess_output.txt'
    drive_debug_path.parent.mkdir(parents=True, exist_ok=True)

    start = time.time()
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    assert proc.stdout is not None
    with local_debug_path.open('w', encoding='utf-8', errors='replace') as f:
        for line in proc.stdout:
            f.write(line)
            if should_live_print(line):
                print(line, end='', flush=True)
            if 'Best metric:' in line:
                mirror_candidate_to_drive(cand_name, work_name)
    proc.wait()
    elapsed_min = (time.time() - start) / 60.0
    shutil.copy2(local_debug_path, drive_debug_path)
    mirror_dir = mirror_candidate_to_drive(cand_name, work_name)
    if proc.returncode != 0:
        tail = local_debug_path.read_text(encoding='utf-8', errors='replace')[-12000:]
        print('\n[error tail]')
        print(tail)
        raise subprocess.CalledProcessError(proc.returncode, cmd, output=tail)
    print(f'\n[done] {cand_name}, elapsed_min={elapsed_min:.2f}')
    return dict(candidate=cand_name, work_name=work_name, mirror_dir=str(mirror_dir), elapsed_min=elapsed_min)

results = []
for cand in QAT_CANDIDATES:
    results.append(run_qat_candidate(cand))
results = [r for r in results if r is not None]
(OUT_DRIVE / 'qat_train_run_summary.json').write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding='utf-8')
print('summary:', OUT_DRIVE / 'qat_train_run_summary.json')

## 7. 대표 checkpoint 수집

학습이 끝나면 후보별 `best.pth`를 한 곳에 모은다. validation metric이 비어 있거나 best가 없으면 마지막 epoch checkpoint를 fallback으로 복사한다.

In [ ]:
PTH_OUT = OUT_DRIVE / 'pths'
PTH_OUT.mkdir(parents=True, exist_ok=True)
collected = []

for cand in QAT_CANDIDATES:
    if not cand.get('enabled', True):
        continue
    cand_name = cand['name']
    work_name = f'{RUN_PREFIX}_{cand_name}'
    run_dir = latest_run_dir(work_name)
    assert run_dir is not None, f'missing run_dir for {cand_name}'
    ckpt_dir = run_dir / 'ckpt'
    assert ckpt_dir.exists(), ckpt_dir

    best = ckpt_dir / 'best.pth'
    epoch_ckpts = sorted([p for p in ckpt_dir.glob('*.pth') if p.stem.isdigit()], key=lambda p: int(p.stem))
    assert best.exists() or epoch_ckpts, f'no checkpoint found for {cand_name}'
    last = epoch_ckpts[-1] if epoch_ckpts else best
    selected = best if best.exists() else last

    selected_dst = PTH_OUT / f'{cand_name}.pth'
    shutil.copy2(selected, selected_dst)

    extra = {}
    if best.exists():
        best_dst = PTH_OUT / f'{cand_name}_best.pth'
        shutil.copy2(best, best_dst)
        extra['best_copy'] = str(best_dst)
    if last.exists():
        last_dst = PTH_OUT / f'{cand_name}_last.pth'
        shutil.copy2(last, last_dst)
        extra['last_copy'] = str(last_dst)

    item = dict(
        candidate=cand_name,
        selected_source=str(selected),
        selected_copy=str(selected_dst),
        best_source=str(best) if best.exists() else None,
        last_source=str(last) if last.exists() else None,
        bytes=selected_dst.stat().st_size,
        **extra,
    )
    collected.append(item)
    print(cand_name, 'selected ->', selected_dst)

manifest = dict(
    created_at=time.strftime('%Y-%m-%d %H:%M:%S'),
    source_best_pth=str(BEST_PTH_SRC),
    dataset=DATASET_NAME,
    qat_epochs=QAT_EPOCHS,
    qat_lr=QAT_LR,
    candidates=QAT_CANDIDATES,
    collected=collected,
    note='candidate.pth is the selected checkpoint for 15/01. _best/_last copies are preserved because training-time validation still uses fallback NMS, while final selection happens in 15/02.',
    next_step='Download outputs_qat_v1/pths or keep in Drive, then run 15/01 export notebook locally.',
)
(OUT_DRIVE / 'pths_manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
print('manifest:', OUT_DRIVE / 'pths_manifest.json')

## 다음 단계

이 노트북이 끝나면 Drive에 아래 파일들이 생긴다.

```text
outputs_qat_v1/pths/qat_layer4_only.pth
outputs_qat_v1/pths/qat_backbone_only.pth
outputs_qat_v1/pths/qat_backbone_neck_only.pth
outputs_qat_v1/pths/qat_full_model.pth
outputs_qat_v1/pths_manifest.json
```

다음 노트북 `01_export_qat_pths_to_onnx_v1.ipynb`에서는 이 네 `.pth`를 일반 CLRKDNet FP32 모델로 로드한 뒤 ONNX로 export한다. 그 다음 `02`에서 ORT static quantization과 11/11b식 의미 보존 평가를 한다.